In [1]:
# BASE 2 — df_traslados (aristas)
# Objetivo

    # Capturar movimientos reales en la red


# Un traslado es:

# dos episodios consecutivos del mismo paciente en hospitales distintos, temporalmente compatibles

# ✅ Entonces tu máscara debería ser:

# ✔️ obligatorio:

    # hospital cambia
    # existe siguiente episodio
    # ventana temporal razonable (tu ±5 días está bien)

# ✔️ opcional:

    # motivo dice traslado → suma confianza, pero no obligatorio
    # Recomendación fuerte

# esta base responde: “¿cómo fluye la red hospitalaria?”

In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import ast
import pandas as pd
import numpy as np
from src.config import *

In [3]:
# 1. CARGA Y RENOMBRE COMPLETO (Recuperado de tu código original)
# ==============================================================================
df_raw = pd.read_excel("../data/pacientes.xlsx")
hospitales = pd.read_csv("../data/hospitales_coordenadas.csv")

dict_comp = dict(zip(hospitales['Nombre Hospital'], hospitales['complejidad']))
hospitales['color_rgb'] = hospitales['color'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df_base = df_raw.rename(columns={
    'Id Hospital': 'hospital_id', 'Nombre Hospital': 'hospital_origen',
    'Id': 'paciente_id', 'Fecha inicio': 'fecha_ingreso', 'Fecha egreso': 'fecha_egreso',
    'Estado al ingreso': 'estado_ingreso', 'Tipo al ingreso': 'tipo_ingreso',
    'Último estado': 'estado_ultimo', 'Último tipo': 'tipo_ultimo',
    'Sexo': 'sexo', 'Edad': 'edad', 'Nivel riesgo clínico': 'riesgo_clinico',
    'Nivel riesgo social': 'riesgo_social', 'Enfermedades preexistentes Covid-19': 'comorbilidades_covid',
    'Enfermedades preexistentes pediatría': 'comorbilidades_pediatria', 'Vacuna': 'vacuna',
    'Cant. dosis': 'cantidad_dosis', '1º dosis': 'fecha_dosis_1', '2º dosis': 'fecha_dosis_2',
    'Buscado en el ministerio': 'validado_ministerio', 'Obra social': 'obra_social',
    'Asistencia Respiratoria Mecánica': 'requiere_arm', 'Motivo': 'motivo_egreso',
    'Operación': 'operacion', 'Última actualización': 'fecha_ultima_actualizacion',
    'Pasó por Críticas': 'paso_criticas', 'Pasó por Intermedias': 'paso_intermedias',
    'Pasó por Generales': 'paso_generales'
}).copy()

df_base['hospital_origen'] = df_base['hospital_origen'].replace({
    'Módulo Hospitalario 11- FV': 'Módulo Hospitalario 11 - FV',
    'Módulo Hospitalario  9 - AB': 'Módulo Hospitalario 9 - AB'
}).str.strip()


df_base['fecha_ingreso'] = pd.to_datetime(df_base['fecha_ingreso'], errors='coerce')
df_base['fecha_egreso'] = pd.to_datetime(df_base['fecha_egreso'], errors='coerce')
df_base['edad'] = pd.to_numeric(df_base['edad'], errors='coerce')
df_base['dias_en_nodo'] = (
    (df_base['fecha_egreso'] - df_base['fecha_ingreso'])
    .dt.days
)

df_base = df_base.sort_values(['paciente_id', 'fecha_ingreso'])

In [4]:
# 2. NORMALIZACION DE MOTIVO DE EGRESO
# ==============================================================================

def normalizar_motivo(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower().strip()
    return x

df_base['motivo_egreso_norm'] = df_base['motivo_egreso'].apply(normalizar_motivo)

# palabras clave de traslado (ajustar según dataset real)
KEYWORDS_TRASLADO = [
    'traslado',
    'derivacion',
    'derivación',
    'referencia',
    'transferencia'
]

def es_traslado(motivo):
    if pd.isna(motivo):
        return 0
    return int(any(k in motivo for k in KEYWORDS_TRASLADO))

df_base['flag_motivo_traslado'] = df_base['motivo_egreso_norm'].apply(es_traslado)

In [5]:
# # 3. DEDUPLICACION TECNICA INGRESO Y EGRESO IGUAL
# # ==============================================================================

# cols_dup = [
#     'paciente_id',
#     'hospital_origen',
#     'fecha_ingreso',
#     'fecha_egreso'
# ]

# df_base['flag_duplicado_exacto'] = df_base.duplicated(subset=cols_dup, keep='first')

# # para construir traslados usamos una version sin duplicados exactos
# df_nodup = df_base[~df_base['flag_duplicado_exacto']].copy()


# ==============================================================================
# 3. DEDUPLICACION POR CERCANIA TEMPORAL (INGRESO Y EGRESO CERCA)
# ==============================================================================

# ordenar bien
df_base = df_base.sort_values([
    'paciente_id',
    'hospital_origen',
    'fecha_ingreso',
    'fecha_egreso'
])

# shift dentro de paciente + hospital
df_base['prev_ingreso'] = df_base.groupby(
    ['paciente_id', 'hospital_origen']
)['fecha_ingreso'].shift(1)

df_base['prev_egreso'] = df_base.groupby(
    ['paciente_id', 'hospital_origen']
)['fecha_egreso'].shift(1)

# diferencias en segundos
df_base['diff_ingreso_seg'] = (
    df_base['fecha_ingreso'] - df_base['prev_ingreso']
).dt.total_seconds().abs()

df_base['diff_egreso_seg'] = (
    df_base['fecha_egreso'] - df_base['prev_egreso']
).dt.total_seconds().abs()

TOL_SEGUNDOS = 60  # podés probar 30, 60, 120
# FLAG DUPLICADO
df_base['flag_duplicado_cercano'] = (
    (df_base['diff_ingreso_seg'] <= TOL_SEGUNDOS) &
    (df_base['diff_egreso_seg'] <= TOL_SEGUNDOS)
)
# CONSTRUCCION SIN ESTOS DUPLICADOS
df_nodup = df_base[~df_base['flag_duplicado_cercano']].copy()

In [6]:
# 4. CONSTRUCCION DE PARES CONSECUTIVOS
# ==============================================================================

df_nodup = df_nodup.sort_values(['paciente_id', 'fecha_ingreso'])

# shift
df_nodup['hospital_destino'] = df_nodup.groupby('paciente_id')['hospital_origen'].shift(-1)
df_nodup['fecha_ingreso_destino'] = df_nodup.groupby('paciente_id')['fecha_ingreso'].shift(-1)
df_nodup['edad_destino'] = df_nodup.groupby('paciente_id')['edad'].shift(-1)

# también motivo del origen ya lo tenemos

In [7]:
# 5. VARIABLES DERIVADAS (DELTA ROBUSTO)
# ==============================================================================

# delta en segundos (base)
df_nodup['delta_segundos'] = (
    df_nodup['fecha_ingreso_destino'] - df_nodup['fecha_egreso']
).dt.total_seconds()

# corrección por tolerancia
df_nodup['delta_segundos_corr'] = df_nodup['delta_segundos'].where(
    df_nodup['delta_segundos'].abs() > TOL_DELTA_SEGUNDOS,
    0
)

# delta en días ENTEROS (evita floats raros)
df_nodup['delta_dias'] = (
    df_nodup['delta_segundos_corr'] / (60 * 60 * 24)
).round().astype('Int64')

# delta edad
df_nodup['delta_edad'] = df_nodup['edad_destino'] - df_nodup['edad']


# ==============================================================================
# 5.B CLASIFICACION TEMPORAL DEL TRASLADO
# ==============================================================================

# en horas (más interpretable que segundos)
df_nodup['delta_horas'] = df_nodup['delta_segundos_corr'] / 3600

# clasificacion
df_nodup['tipo_traslado'] = np.select(
    [
        df_nodup['delta_segundos_corr'] == 0,
        (df_nodup['delta_horas'] > 0) & (df_nodup['delta_horas'] <= 24),
        df_nodup['delta_horas'] > 24
    ],
    [
        'inmediato',   # mismo evento (<= 5 min)
        'rapido',      # dentro del mismo día
        'diferido'     # más de 1 día
    ],
    default='otro'  # incluye negativos raros o edge cases
)

In [8]:
#  6. FLAGS DE CALIDAD
# ==============================================================================

df_nodup['flag_mismo_hospital'] = (
    df_nodup['hospital_origen'] == df_nodup['hospital_destino']
)

df_nodup['flag_fechas_faltantes'] = (
    df_nodup['fecha_egreso'].isna() |
    df_nodup['fecha_ingreso_destino'].isna()
)

df_nodup['flag_overlap'] = df_nodup['delta_dias'] < 0
df_nodup['flag_overlap_extremo'] = df_nodup['delta_dias'] < -2

df_nodup['flag_gap_corto'] = df_nodup['delta_dias'].between(0, 2)
df_nodup['flag_gap_medio'] = df_nodup['delta_dias'].between(3, 5)
df_nodup['flag_gap_largo'] = df_nodup['delta_dias'] > 5

df_nodup['flag_edad_inconsistente'] = df_nodup['delta_edad'].abs() > 2

df_nodup['flag_inmediato'] = df_nodup['tipo_traslado'] == 'inmediato'
df_nodup['flag_diferido'] = df_nodup['tipo_traslado'] == 'diferido'

In [9]:
#  7. FILTROS BASE (SIN MOTIVO)
# ==============================================================================

cond_base = (
    (df_nodup['hospital_destino'].notna()) &
    (~df_nodup['flag_mismo_hospital']) &
    (~df_nodup['flag_fechas_faltantes']) &
    (df_nodup['delta_dias'] >= -5) &
    (df_nodup['delta_dias'] <= 30) &
    (~df_nodup['flag_edad_inconsistente'])
)

df_pares_validos = df_nodup[cond_base].copy()

In [10]:
# 8. DF TRASLADOS LOOSE
# ==============================================================================

df_traslados_loose = df_pares_validos.copy()

In [11]:
# 9. DF TRASLADOS STRICT
# ==============================================================================

cond_motivo = df_pares_validos['flag_motivo_traslado'] == 1

df_traslados_strict = df_pares_validos[cond_motivo].copy()

In [12]:
# 10. COLUMNAS FINALES
# ==============================================================================

cols_finales = [
    'paciente_id',
    'hospital_origen',
    'hospital_destino',
    'fecha_egreso',
    'fecha_ingreso_destino',
    'delta_dias',
    'edad',
    'edad_destino',
    'delta_edad',
    'flag_motivo_traslado',
    'flag_overlap',
    'flag_overlap_extremo',
    'flag_gap_corto',
    'flag_gap_medio',
    'flag_gap_largo',
    'flag_edad_inconsistente',
    'delta_segundos_corr',
    'tipo_traslado',
    'delta_horas',
    'flag_inmediato',
    'flag_diferido'
]

df_traslados_loose = df_traslados_loose[cols_finales].rename(columns={
    'fecha_egreso': 'fecha_egreso_origen'
})

df_traslados_strict = df_traslados_strict[cols_finales].rename(columns={
    'fecha_egreso': 'fecha_egreso_origen'
})

In [ ]:
df_traslados_loose.to_excel("../data/final_data/df_traslados_loose.xlsx", index=False)

df_traslados_strict.to_excel("../data/final_data/df_traslados_strict.xlsx", index=False)

df_nodup.to_parquet("../data/debug/df_nodup_debug.parquet")

ArrowInvalid: ("Could not convert 'A' with type str: tried to convert to int64", 'Conversion failed for column paciente_id with type object')

### verificaciones y explroacion

In [13]:
# SANITY CHECKS GENERALES
# ==============================================================================

print("=== TAMAÑOS ===")
print("Base original:", len(df_base))
print("Sin duplicados:", len(df_nodup))
print("Traslados loose:", len(df_traslados_loose))
print("Traslados strict:", len(df_traslados_strict))


print("\n=== DELTA DIAS (LOOSE) ===")
print(df_traslados_loose['delta_dias'].describe())

print("\n=== DELTA DIAS (STRICT) ===")
print(df_traslados_strict['delta_dias'].describe())


print("\n=== DELTA EDAD ===")
print(df_traslados_loose['delta_edad'].describe())


print("\n=== PROPORCION CON MOTIVO ===")
print(df_traslados_loose['flag_motivo_traslado'].value_counts(normalize=True))


print("\n=== EJEMPLOS RANDOM ===")
display(df_traslados_loose.sample(5))


print("\n=== CHEQUEO CLAVE: hospitales distintos ===")
print((df_traslados_loose['hospital_origen'] == df_traslados_loose['hospital_destino']).sum())


print("\n=== CHEQUEO EDAD ===")
print((df_traslados_loose['delta_edad'].abs() > 2).sum())


print("\n=== OVERLAPS ===")
print(df_traslados_loose['flag_overlap'].value_counts())


print("\n=== TOP FLUJOS A->B ===")
print(
    df_traslados_loose
    .groupby(['hospital_origen', 'hospital_destino'])
    .size()
    .sort_values(ascending=False)
    .head(10)
)

=== TAMAÑOS ===
Base original: 29697
Sin duplicados: 29683
Traslados loose: 1857
Traslados strict: 1670

=== DELTA DIAS (LOOSE) ===
count      1857.0
mean    -0.208939
std      1.578228
min          -5.0
25%           0.0
50%           0.0
75%           0.0
max          30.0
Name: delta_dias, dtype: Float64

=== DELTA DIAS (STRICT) ===
count      1670.0
mean     0.038922
std      1.315545
min          -5.0
25%           0.0
50%           0.0
75%           0.0
max          30.0
Name: delta_dias, dtype: Float64

=== DELTA EDAD ===
count    1826.000000
mean        0.006024
std         0.142259
min        -2.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         2.000000
Name: delta_edad, dtype: float64

=== PROPORCION CON MOTIVO ===
flag_motivo_traslado
1    0.8993
0    0.1007
Name: proportion, dtype: float64

=== EJEMPLOS RANDOM ===


,paciente_id,hospital_origen,hospital_destino,fecha_egreso_origen,fecha_ingreso_destino,delta_dias,edad,edad_destino,delta_edad,flag_motivo_traslado,...,flag_overlap_extremo,flag_gap_corto,flag_gap_medio,flag_gap_largo,flag_edad_inconsistente,delta_segundos_corr,tipo_traslado,delta_horas,flag_inmediato,flag_diferido
6627,SQ59,Módulo Hospitalario 11 - FV,UPA 11 - FV,2021-01-26 12:11:10,2021-01-22 12:11:26,-4,46.0,46.0,0.0,0,...,True,False,False,False,False,-345584.0,otro,-95.995556,False,False
23184,XB50,UPA 5 - AB,Módulo Hospitalario 9 - AB,2021-05-13 12:11:25,2021-05-13 12:11:21,0,39.0,39.0,0.0,1,...,False,True,False,False,False,0.0,inmediato,0.000000,True,False
19850,YL35,Módulo Hospitalario 9 - AB,UPA 5 - AB,2021-12-30 12:11:21,2021-12-29 12:11:25,-1,46.0,46.0,0.0,1,...,False,False,False,False,False,-86396.0,otro,-23.998889,False,False
12387,CA93,Lucio Meléndez,Módulo Hospitalario 9 - AB,2020-08-20 12:11:16,2020-08-20 12:11:21,0,58.0,58.0,0.0,1,...,False,True,False,False,False,0.0,inmediato,0.000000,True,False
6355,EI64,Módulo Hospitalario 11 - FV,UPA 11 - FV,2020-08-11 12:11:10,2020-08-11 12:11:26,0,55.0,55.0,0.0,1,...,False,True,False,False,False,0.0,inmediato,0.000000,True,False



=== CHEQUEO CLAVE: hospitales distintos ===
0

=== CHEQUEO EDAD ===
0

=== OVERLAPS ===
flag_overlap
False    1622
True      235
Name: count, dtype: Int64

=== TOP FLUJOS A->B ===
hospital_origen              hospital_destino           
UPA 5 - AB                   Módulo Hospitalario 9 - AB     477
UPA 11 - FV                  Módulo Hospitalario 11 - FV    328
Mi Pueblo                    Módulo Hospitalario 11 - FV    269
Módulo Hospitalario 9 - AB   UPA 5 - AB                     139
Módulo Hospitalario 11 - FV  UPA 11 - FV                     99
Lucio Meléndez               Módulo Hospitalario 9 - AB      90
UPA 5 - AB                   Lucio Meléndez                  73
Lucio Meléndez               UPA 5 - AB                      72
UPA 17 - QU                  Módulo Hospitalario 10 - QU     56
Módulo Hospitalario 9 - AB   Lucio Meléndez                  28
dtype: int64


In [14]:
print("\n=== DISTRIBUCION POR DELTA ===")
print(df_traslados_loose['delta_dias'].value_counts().sort_index())


=== DISTRIBUCION POR DELTA ===
delta_dias
-5      27
-4      47
-3      41
-2      47
-1      73
0     1575
1       20
2        5
3        1
4        5
5        1
6        4
7        2
8        1
9        1
10       1
11       1
12       2
16       1
21       1
30       1
Name: count, dtype: Int64


In [15]:
print("\n=== TIPOS DE TRASLADO ===")
print(df_traslados_loose['tipo_traslado'].value_counts(normalize=True))


=== TIPOS DE TRASLADO ===
tipo_traslado
inmediato    0.848142
otro         0.126548
diferido     0.016155
rapido       0.009155
Name: proportion, dtype: float64
